# Side analysis: mutually reverting editor pairs

**Status.** Side analysis, written with Claude Code. It is **not** part of the checked numbers: nothing here is in `docs/final/results.json`, the report, the slides or the dashboard. Results are saved to `notebooks/side/mutual_reverts_results.json`.

**Question.** On pages where two editors revert each other, do revert labels become known sooner or later than elsewhere?

**Definitions** (after Yasseri, Sumi, Rung, Kornai, Kertesz 2012, *Dynamics of conflicts in Wikipedia*, PLoS ONE 7(6) e38869).
- *Mutual pair*: two editors A and B on one page where A reverted B and B reverted A.
- *M*: E x the sum over the page's mutual pairs, leaving out the largest, of min(N_A, N_B). E = editors in mutual pairs, N = the editor's edit count on the page.

**Where this differs from the paper.**
- N is counted only inside the data window (3 days, 20 days or 7 weeks), not over the page's whole history. It is a "window M".
- Stream cohorts take reverts from MediaWiki revert events (undo, rollback, manual). The dump takes them from identical page content (SHA-1).
- A pair is seen only if both reverted edits fall inside the window.

**Data.** Silver revert-link tables (one row per reverted revision, with the reverting and the reverted editor) and `silver/history_slim` (dump, September and October 2026). English Wikipedia, article namespace.

**Privacy.** Editor keys in the lake are not hashed. They are used here only to test whether two rows have the same editor. No key, name, comment or page title is printed or saved; every output is a count or a share.

In [1]:
import json, os, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'side' else Path.cwd()
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / 'notebooks' / 'side'))
os.environ.setdefault('TRUTHLAG_HW2_DIR', str(REPO / 'ec-hw2'))
os.environ['TZ'] = 'UTC'
import numpy as np, pandas as pd
import pyarrow.compute as pc
from deltalake import DeltaTable
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
import mutual_reverts_lib as mr
from etl.hypotheses import poisson_ci, poisson_diff_ci, SEED, B, BURST_MIN
from etl.hw2 import config
SILVER = REPO / 'data' / 'lake' / 'silver'
H = config.LIVE_FOLLOWUP_S
rng = np.random.default_rng(SEED)
FACTS = {}
mr.self_test()
TABLES = ['history_revert_links', 'live_revert_links', 'live_ext_revert_links', 'history_revisions', 'live_revisions', 'live_ext_revisions', 'history_slim']
VERSIONS = {t: DeltaTable(str(SILVER / t)).version() for t in TABLES}
print('Delta table versions read:', VERSIONS)
print('bootstrap draws:', B, '| seed:', SEED, '| follow-up window:', H, 's')

self-test ok
Delta table versions read: {'history_revert_links': 1, 'live_revert_links': 3, 'live_ext_revert_links': 1, 'history_revisions': 1, 'live_revisions': 3, 'live_ext_revisions': 1, 'history_slim': 2}
bootstrap draws: 2000 | seed: 226 | follow-up window: 345600 s


## 1. How many mutual pairs

`history` is the dump cohort (edits of Sep 8-10). `live_ext` is the stream cohort (edits of Oct 20 - Nov 8). `live` (Oct 20-22) is the first three days of `live_ext`, so it is not an independent sample; it shows the effect of a shorter window.

Rows of the table: the base count, then the same count under stricter rules.

In [2]:
LCOLS = ['reverted_rev_id', 'reverting_rev_id', 'page_id', 'author_key', 'author_kind', 'reverter_key', 'reverter_kind', 'T_s', 'K_s', 'by_other', 'within_followup', 'range_size']
COHORTS = {'history': ('history_revert_links', 'history_revisions'), 'live': ('live_revert_links', 'live_revisions'), 'live_ext': ('live_ext_revert_links', 'live_ext_revisions')}
links, revs = {}, {}
for c, (lt, rt) in COHORTS.items():
    links[c] = DeltaTable(str(SILVER / lt)).to_pandas(columns=LCOLS)
    revs[c] = DeltaTable(str(SILVER / rt)).to_pandas(columns=['rev_id', 'page_id', 'user_key', 'page_edits_prev_1h'])
    assert not links[c].reverted_rev_id.duplicated().any()
    assert (links[c].by_other.fillna(False).astype(bool) == (links[c].author_key.notna() & links[c].reverter_key.notna() & (links[c].author_key != links[c].reverter_key))).all()

VARIANTS = [('base: reverter is not the author', {}), ('no bots', {'no_bots': True}), ('registered editors only', {'registered_only': True}),
            ('revert undid exactly one revision', {'single_only': True}), ('revert inside the 4-day follow-up', {'followup_only': True})]
rows = []
for c in COHORTS:
    for name, kw in VARIANTS:
        if kw.get('single_only') and links[c].range_size.isna().all():
            continue   # the dump has no range size
        r = mr.counts(links[c], **kw)
        rows.append({'cohort': c, 'rule': name, **r})
        FACTS[f'{c}.{name}'] = r
counts_df = pd.DataFrame(rows).drop(columns=['link_rows'])
for c in COHORTS:
    FACTS[f'{c}.cohort_pages'] = int(revs[c].page_id.nunique())
    print(c, '| link rows', len(links[c]), '| cohort pages', FACTS[f'{c}.cohort_pages'])
counts_df

history | link rows 19858 | cohort pages 165415
live | link rows 19513 | cohort pages 155938
live_ext | link rows 122192 | cohort pages 1238130


,cohort,rule,usable_links,directed_pairs,mutual_pairs,mutual_pages,pages_with_a_revert_by_another_editor,mutual_pages_pct_of_reverted_pages,pages_with_one_pair,pages_with_2plus_pairs
0,history,base: reverter is not the author,18020,13353,462,397,10465,3.79,356,41
1,history,no bots,17123,12516,437,382,10015,3.81,348,34
2,history,registered editors only,7058,4851,194,171,4105,4.17,156,15
3,history,revert inside the 4-day follow-up,17868,13269,462,397,10419,3.81,356,41
4,live,base: reverter is not the author,17885,13613,510,438,10600,4.13,392,46
5,live,no bots,17414,13154,503,434,10300,4.21,388,46
6,live,registered editors only,7028,4973,225,215,4098,5.25,207,8
7,live,revert undid exactly one revision,11310,10881,405,354,9035,3.92,319,35
8,live,revert inside the 4-day follow-up,16265,12441,478,406,9664,4.20,360,46
9,live_ext,base: reverter is not the author,110894,83347,3619,2831,55512,5.10,2377,454


## 2. Who is in the pairs, and how many pairs per page

Pairs are split by the account kind of the two editors. The data does not say what a pair disagreed about. Temporary accounts can change key over time, so pairs with them are undercounted.

In [3]:
kind_rows = []
for c in COHORTS:
    m = mr.mutual_pairs(mr.usable_links(links[c]))
    vc = m.pair_kind.value_counts()
    for k, v in vc.items():
        kind_rows.append({'cohort': c, 'pair kind': k, 'mutual pairs': int(v), 'share %': round(100 * v / len(m), 1)})
    FACTS[f'{c}.pair_kinds'] = {k: int(v) for k, v in vc.items()}
pd.DataFrame(kind_rows).pivot(index='pair kind', columns='cohort', values='mutual pairs').fillna(0).astype(int)[list(COHORTS)]

cohort,history,live,live_ext
pair kind,,,
bot vs bot,0,0,5
bot vs registered,4,2,70
bot vs temporary,21,5,116
registered vs registered,194,225,1608
registered vs temporary,235,274,1764
temporary vs temporary,8,4,56


In [4]:
m_rows = []
for c in COHORTS:
    m = mr.mutual_pairs(mr.usable_links(links[c]))
    n_edits = revs[c].groupby(['page_id', 'user_key']).size()
    M, missing = mr.window_m(m, n_edits)
    pos = M[M > 0]
    r = {'cohort': c, 'mutual pages': int(len(M)), 'pages with window M = 0': int((M == 0).sum()), 'pages with window M > 0': int(len(pos)),
         'median M where > 0': float(pos.median()) if len(pos) else None, 'largest M': float(M.max()),
         'pairs where an editor has no counted edit (N set to 1)': missing}
    m_rows.append(r); FACTS[f'{c}.window_M'] = r
print('Window M is 0 on any page with a single mutual pair, because the formula leaves out the largest pair.')
pd.DataFrame(m_rows)

Window M is 0 on any page with a single mutual pair, because the formula leaves out the largest pair.


,cohort,mutual pages,pages with window M = 0,pages with window M > 0,median M where > 0,largest M,pairs where an editor has no counted edit (N set to 1)
0,history,397,356,41,6.0,170.0,0
1,live,438,392,46,4.0,187.0,0
2,live_ext,2831,2377,454,6.0,2080.0,0


## 3. Truth lag on pages with a mutual pair (flag from the same window)

E(L) is the share of eventual reverts **not yet known** after waiting L. Lower means known sooner. Intervals are 95% Poisson-bootstrap intervals over pages, with the project's own code (`etl.hypotheses.poisson_ci`).

Three page groups: pages with a mutual pair; other pages with two or more revert events by another editor (a mutual pair needs at least two revert events; a rollback of several edits is one event); all remaining pages (one such event, or only self-reverts). The difference column compares against the two-or-more group. The flag uses every link in the table, including reverts after the 4-day follow-up.

The second table shows how the groups differ in who wrote the reverted edits and in the share of self-reverts. E(L) differs by both, so a difference between groups cannot be put down to the flag alone.

**Read with care.** The flag is built from the same reverts that are being measured, so this is a description, not a test. For `history`, the dump gives the time to revert; only the stream cohorts measure when a label became known.

In [5]:
def frame_for(c):
    l = links[c]
    f = l[(l.T_s >= 0) & (l.T_s <= H)].merge(revs[c][['rev_id', 'page_edits_prev_1h']], left_on='reverted_rev_id', right_on='rev_id')
    return f.assign(known_after_s=f.T_s.astype(float) + f.K_s.fillna(0).astype(float))

def composition(f, group_of_page, default):
    g = f.assign(group=f.page_id.map(group_of_page).fillna(default), temp=f.author_kind == 'temporary', reg=f.author_kind == 'registered',
                 bot=f.author_kind == 'bot', self_rev=~f.by_other.fillna(False).astype(bool))
    t = g.groupby('group').agg(reverts=('temp', 'size'), temporary_author_pct=('temp', 'mean'), registered_author_pct=('reg', 'mean'),
                               bot_author_pct=('bot', 'mean'), self_revert_pct=('self_rev', 'mean'))
    t[t.columns[1:]] = (100 * t[t.columns[1:]]).round(1)
    return t.reset_index()
COMP_COLS = ['cohort', 'group', 'reverts', 'temporary_author_pct', 'registered_author_pct', 'bot_author_pct', 'self_revert_pct']

G3 = ['mutual pair', 'no mutual pair, 2+ revert events', 'all other pages']
def groups_in_window(c):
    d = mr.usable_links(links[c])
    mp = set(mr.mutual_pairs(d).page_id)
    n = d.groupby('page_id').reverting_rev_id.nunique()
    g = pd.Series(G3[1], index=n.index[n >= 2])
    g = pd.concat([g[~g.index.isin(mp)], pd.Series(G3[0], index=sorted(mp))])
    return g

frames = {c: frame_for(c) for c in COHORTS}
split_rows, comp_rows = [], []
for c in COHORTS:
    res = mr.el_split(frames[c], groups_in_window(c), G3, G3[1], G3[2], rng, poisson_ci, poisson_diff_ci)
    FACTS[f'{c}.E_by_in_window_flag'] = res
    split_rows += [{'cohort': c, 'measure': 'time to revert' if c == 'history' else 'time until the label is known', **r} for r in res]
    comp = composition(frames[c], groups_in_window(c), G3[2]); FACTS[f'{c}.composition_in_window_groups'] = comp.to_dict('records')
    comp_rows.append(comp.assign(cohort=c))
    print(c, '| eventual reverts in the frame:', len(frames[c]), '| pages:', frames[c].page_id.nunique())
pd.DataFrame(split_rows)

history | eventual reverts in the frame: 17646 | pages: 10364


live | eventual reverts in the frame: 17818 | pages: 10471


live_ext | eventual reverts in the frame: 112937 | pages: 56531


,cohort,measure,L,group,pages,reverts,E,E_lo,E_hi,diff_vs_ref,diff_lo,diff_hi,diff_interval_excludes_0
0,history,time to revert,1 h,mutual pair,397,1983,0.3777,0.3329,0.4240,0.0551,0.0020,0.1086,True
1,history,time to revert,1 h,"no mutual pair, 2+ revert events",1235,4039,0.3226,0.2972,0.3486,NaN,NaN,NaN,NaN
2,history,time to revert,1 h,all other pages,8732,11624,0.4966,0.4837,0.5095,0.1740,0.1435,0.2039,True
3,history,time to revert,1 d,mutual pair,397,1983,0.0605,0.0462,0.0765,0.0110,-0.0083,0.0309,False
4,history,time to revert,1 d,"no mutual pair, 2+ revert events",1235,4039,0.0495,0.0383,0.0623,NaN,NaN,NaN,NaN
5,history,time to revert,1 d,all other pages,8732,11624,0.1401,0.1288,0.1522,0.0906,0.0738,0.1077,True
6,live,time until the label is known,1 h,mutual pair,438,2350,0.3740,0.3266,0.4233,0.0495,-0.0018,0.1021,False
7,live,time until the label is known,1 h,"no mutual pair, 2+ revert events",1229,4037,0.3245,0.2990,0.3506,NaN,NaN,NaN,NaN
8,live,time until the label is known,1 h,all other pages,8804,11431,0.4639,0.4489,0.4783,0.1394,0.1109,0.1686,True
9,live,time until the label is known,1 d,mutual pair,438,2350,0.0545,0.0419,0.0697,-0.0040,-0.0235,0.0161,False


In [6]:
pd.concat(comp_rows)[COMP_COLS]

,cohort,group,reverts,temporary_author_pct,registered_author_pct,bot_author_pct,self_revert_pct
0,history,all other pages,11624,56.2,43.0,0.8,12.6
1,history,mutual pair,1983,38.0,60.1,1.9,3.0
2,history,"no mutual pair, 2+ revert events",4039,63.8,35.6,0.5,4.4
0,live,all other pages,11431,55.6,43.4,1.0,11.1
1,live,mutual pair,2350,42.9,56.6,0.5,5.6
2,live,"no mutual pair, 2+ revert events",4037,59.5,40.1,0.4,3.9
0,live_ext,all other pages,57207,52.9,44.8,2.3,14.1
1,live_ext,mutual pair,20271,40.2,58.0,1.8,4.8
2,live_ext,"no mutual pair, 2+ revert events",35459,60.0,39.2,0.8,4.3


## 4. Flag pages from an earlier period

Pages are flagged from the dump for **Sep 1 - Oct 19** (both the reverted edit and the revert before Oct 20). Truth lag is then measured on stream edits from **Oct 20** on. The flag uses no information from the measured period.

Groups: pages with a mutual pair in the earlier period; other pages with two or more revert events by another editor in the earlier period; all remaining pages. The last group includes pages with no edit at all in the earlier period (count printed below). The difference column compares against the two-or-more group.

In [7]:
CUT = pd.Timestamp('2026-10-20', tz='UTC')
assert all(DeltaTable(str(SILVER / t)).to_pandas(columns=['reverted_ts']).reverted_ts.min() >= CUT for t in ('live_revert_links', 'live_ext_revert_links'))
ds = DeltaTable(str(SILVER / 'history_slim')).to_pyarrow_dataset()
hs = ds.to_table(filter=(pc.field('snapshot') == '2026-10') & (pc.field('page_namespace') == 0),
                 columns=['revision_id', 'page_id', 'event_timestamp', 'revision_first_identity_reverting_revision_id', 'event_user_id', 'event_user_text', 'event_user_is_bot_by']).to_pandas()
assert not hs.revision_id.duplicated().any()
uid = hs.event_user_id.astype('Int64').astype('string')
hs['key'] = ('u:' + uid).where(hs.event_user_id.notna(), 't:' + hs.event_user_text)
is_bot = hs.event_user_is_bot_by.fillna('').str.len() > 0
# simpler rule than the lake's: any bot flag is a bot, any name starting with ~ is a temporary account
hs['kind'] = np.select([is_bot, hs.event_user_text.fillna('').str.startswith('~'), hs.event_user_id.notna()], ['bot', 'temporary', 'registered'], 'unknown')
print('dump revisions, article namespace, snapshot 2026-10:', len(hs), '| from', hs.event_timestamp.min(), 'to', hs.event_timestamp.max())
print('editor kinds:', hs.kind.value_counts().to_dict())

rv = hs[hs.revision_first_identity_reverting_revision_id.notna() & (hs.event_timestamp < CUT)]
look = hs.set_index('revision_id')[['key', 'kind', 'event_timestamp']].rename(columns={'key': 'reverter_key', 'kind': 'reverter_kind', 'event_timestamp': 'reverting_ts'})
pre = rv.rename(columns={'key': 'author_key', 'kind': 'author_kind'}).join(look, on='revision_first_identity_reverting_revision_id', how='left')
n_all = len(pre); n_missing = int(pre.reverting_ts.isna().sum()); n_late = int((pre.reverting_ts >= CUT).sum())
pre = pre[pre.reverting_ts < CUT].assign(range_size=np.nan, within_followup=True)
pre_use = mr.usable_links(pre)
pre_m = mr.mutual_pairs(pre_use)
pre_counts = mr.counts(pre)
n_pre = hs[hs.event_timestamp < CUT].groupby(['page_id', 'key']).size()
M, missing = mr.window_m(pre_m, n_pre)
FACTS['pre_period'] = {'window': 'reverted edit and revert both before 2026-10-20, dump snapshot 2026-10', 'first_dump_edit': str(hs.event_timestamp.min()),
                       'reverted_edits_before_cut': n_all, 'reverting_revision_not_in_table': n_missing, 'reverted_after_cut_dropped': n_late,
                       **pre_counts, 'pair_kinds': {k: int(v) for k, v in pre_m.pair_kind.value_counts().items()},
                       'pages_with_window_M_gt_0': int((M > 0).sum()), 'largest_window_M': float(M.max())}
print('reverted edits before Oct 20:', n_all, '| reverting revision not in the table:', n_missing, '| reverted on or after Oct 20 (dropped):', n_late)
print({k: v for k, v in pre_counts.items() if k != 'link_rows'})
print('pair kinds:', FACTS['pre_period']['pair_kinds'])
print('pages with window M > 0:', FACTS['pre_period']['pages_with_window_M_gt_0'], 'of', len(M), '| largest window M:', FACTS['pre_period']['largest_window_M'])

dump revisions, article namespace, snapshot 2026-10: 7161897 | from 2026-09-01 00:00:00+00:00 to 2026-11-01 01:30:05+00:00
editor kinds: {'registered': 4818449, 'bot': 1591112, 'temporary': 751990, 'unknown': 346}


reverted edits before Oct 20: 339663 | reverting revision not in the table: 0 | reverted on or after Oct 20 (dropped): 16246
{'usable_links': 290983, 'directed_pairs': 210830, 'mutual_pairs': 9634, 'mutual_pages': 7615, 'pages_with_a_revert_by_another_editor': 122809, 'mutual_pages_pct_of_reverted_pages': 6.2, 'pages_with_one_pair': 6364, 'pages_with_2plus_pairs': 1251}
pair kinds: {'registered vs temporary': 4562, 'registered vs registered': 4201, 'bot vs registered': 504, 'bot vs temporary': 273, 'temporary vs temporary': 93, 'bot vs bot': 1}
pages with window M > 0: 1251 of 7615 | largest window M: 722.0


In [8]:
P3 = ['mutual pair before Oct 20', 'no mutual pair, 2+ revert events before Oct 20', 'all other pages']
n = pre_use.groupby('page_id').revision_first_identity_reverting_revision_id.nunique()
pre_group = pd.Series(P3[1], index=n.index[n >= 2])
pre_group = pd.concat([pre_group[~pre_group.index.isin(set(pre_m.page_id))], pd.Series(P3[0], index=sorted(set(pre_m.page_id)))])
assert not pre_group.index.duplicated().any()
pre_rows, pre_comp = [], []
edited_before = set(hs[hs.event_timestamp < CUT].page_id)
for c in ('live_ext', 'live'):
    res = mr.el_split(frames[c], pre_group, P3, P3[1], P3[2], rng, poisson_ci, poisson_diff_ci)
    FACTS[f'{c}.E_by_pre_period_flag'] = res
    pre_rows += [{'cohort': c, **r} for r in res]
    comp = composition(frames[c], pre_group, P3[2]); FACTS[f'{c}.composition_pre_period_groups'] = comp.to_dict('records')
    pre_comp.append(comp.assign(cohort=c))
    fp = set(frames[c].page_id)
    FACTS[f'{c}.frame_pages_with_no_dump_edit_before_cut'] = len(fp - edited_before)
    print(c, '| pages in the frame:', len(fp), '| of those with no edit in the dump before Oct 20 (all in the last group):', len(fp - edited_before))
pd.DataFrame(pre_rows)

live_ext | pages in the frame: 56531 | of those with no edit in the dump before Oct 20 (all in the last group): 16933


live | pages in the frame: 10471 | of those with no edit in the dump before Oct 20 (all in the last group): 2446


,cohort,L,group,pages,reverts,E,E_lo,E_hi,diff_vs_ref,diff_lo,diff_hi,diff_interval_excludes_0
0,live_ext,1 h,mutual pair before Oct 20,1735,7594,0.4272,0.3862,0.4695,0.0355,-0.0095,0.0849,False
1,live_ext,1 h,"no mutual pair, 2+ revert events before Oct 20",6911,19944,0.3916,0.3707,0.4113,NaN,NaN,NaN,NaN
2,live_ext,1 h,all other pages,47885,85399,0.4161,0.4098,0.4226,0.0245,0.0030,0.0467,True
3,live_ext,1 d,mutual pair before Oct 20,1735,7594,0.0762,0.0650,0.0897,0.0080,-0.0058,0.0235,False
4,live_ext,1 d,"no mutual pair, 2+ revert events before Oct 20",6911,19944,0.0683,0.0615,0.0754,NaN,NaN,NaN,NaN
5,live_ext,1 d,all other pages,47885,85399,0.1029,0.0984,0.1078,0.0346,0.0264,0.0430,True
6,live,1 h,mutual pair before Oct 20,550,1430,0.4510,0.3910,0.5131,0.0477,-0.0163,0.1092,False
7,live,1 h,"no mutual pair, 2+ revert events before Oct 20",1813,3265,0.4034,0.3753,0.4322,NaN,NaN,NaN,NaN
8,live,1 h,all other pages,8108,13123,0.4214,0.4065,0.4377,0.0180,-0.0144,0.0515,False
9,live,1 d,mutual pair before Oct 20,550,1430,0.1217,0.0866,0.1598,0.0295,-0.0102,0.0730,False


In [9]:
pd.concat(pre_comp)[COMP_COLS]

,cohort,group,reverts,temporary_author_pct,registered_author_pct,bot_author_pct,self_revert_pct
0,live_ext,all other pages,85399,51.9,46.1,2.0,10.4
1,live_ext,mutual pair before Oct 20,7594,48.2,50.5,1.2,7.6
2,live_ext,"no mutual pair, 2+ revert events before Oct 20",19944,58.7,40.4,0.9,5.5
0,live,all other pages,13123,53.6,45.5,0.9,9.4
1,live,mutual pair before Oct 20,1430,48.5,50.7,0.8,7.9
2,live,"no mutual pair, 2+ revert events before Oct 20",3265,62.6,37.1,0.3,6.4


## 5. Overlap with the burst flag of H3

H3 calls an edit a burst edit when its page had at least 10 edits in the hour before. A page counts here as a burst page if it has at least one reverted burst edit in the frame.

In [10]:
ov = []
for c in COHORTS:
    f = frames[c]
    burst_pages = set(f[f.page_edits_prev_1h.fillna(-1) >= BURST_MIN].page_id)
    mut_pages = set(mr.mutual_pairs(mr.usable_links(links[c])).page_id) & set(f.page_id)
    r = {'cohort': c, 'pages in the frame': int(f.page_id.nunique()), 'burst pages': len(burst_pages), 'mutual-pair pages': len(mut_pages), 'both': len(burst_pages & mut_pages)}
    if c != 'history':
        pm = set(pre_m.page_id) & set(f.page_id)
        r.update({'pages with a mutual pair before Oct 20': len(pm), 'of those, also a mutual pair in the window': len(pm & mut_pages)})
    ov.append(r); FACTS[f'{c}.overlap'] = r
pd.DataFrame(ov)

,cohort,pages in the frame,burst pages,mutual-pair pages,both,pages with a mutual pair before Oct 20,"of those, also a mutual pair in the window"
0,history,10364,83,397,19,NaN,NaN
1,live,10471,70,438,24,550.0,55.0
2,live_ext,56531,506,2829,183,1735.0,283.0


In [11]:
out_path = REPO / 'notebooks' / 'side' / 'mutual_reverts_results.json'
out_path.write_text(json.dumps({'note': 'Side analysis. Not part of docs/final/results.json or any checked number.',
                                'delta_versions': VERSIONS, 'bootstrap_draws': B, 'seed': SEED, 'facts': FACTS}, indent=1, default=str))
print('wrote', out_path.relative_to(REPO), 'with', len(FACTS), 'entries')

wrote notebooks/side/mutual_reverts_results.json with 39 entries


## Limits

- Side analysis: the numbers are not in `results.json` and are not covered by `scripts/check_numbers.py`.
- The flag in section 3 is built from the reverts it is compared on. Section 4 avoids this.
- `live` is part of `live_ext`. The independent samples are the September dump cohort and the October-November stream cohort.
- A pair is found only when both reverted edits are inside the window, so pairs are undercounted, more so in short windows.
- Temporary accounts can hold several keys over time, so pairs that involve them are undercounted.
- A rollback of several edits makes a link to each reverted author. The "exactly one revision" row in section 1 shows the count without these.
- Window M uses edit counts inside the window only and is 0 on every page with a single pair. It is not comparable with published values of M.
- Partial reverts that carry no revert tag and do not restore identical content are not seen.
- Section 3 compares groups in three cohorts (two of them overlap) at two waits, and section 4 in two cohorts at two waits, with no correction for multiple comparisons.
- The page groups differ in the kind of author and in the share of self-reverts (tables in sections 3 and 4), and E(L) differs by both. A group difference is not an effect of the flag.
- For `history`, E(L) is the share not yet reverted after L (time to revert). The dump has no label delay.
- The flag in section 4 misses pairs that began before Sep 1 or that straddle Oct 20, and pages with no edit before Oct 20 can only be in the last group.
- Section 4 labels editor kinds from the dump with a simpler rule than the lake (any bot flag is a bot; any name starting with ~ is a temporary account). Pair and page counts do not use the kind.
- Interval ends come from a finite number of bootstrap draws (printed in the first cell), so they carry some Monte Carlo noise.
- No edit text is used. A mutual pair shows that two editors undid each other, not what they disagreed about.